In [ ]:
import os
import torch
import warnings
import chunking as C
from tqdm import tqdm
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from constants import SEED, device
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
warnings.filterwarnings("ignore")

### Extract every expert checkpoint, the backbone is handled separately

In [23]:

rows = []
models_path = Path('res_models')

EXCLUDE_NAMES = {'backbone_a.pt'}

ckpt_paths = sorted(
    p for p in models_path.rglob('*.pt')
    if 'resume' not in p.name and p.name not in EXCLUDE_NAMES
)

for path in tqdm(ckpt_paths, desc='extracting'):
    ck = torch.load(path, map_location='cpu')

    '''strict load so a checkpoint that does not match the architecture fails here'''
    model = resnet20(num_classes=100)
    model.load_state_dict(ck['state_dict'], strict=True)
    sd = model.state_dict()

    chunks, mask, seq_index, meta = C.extract_model(
        sd, ck['model_id'], checkpoint_path=str(path), split_id=ck['split_id'],
        seed=ck['seed'], epoch=ck['epoch'], init_group=ck['init_group']
    )

    C.verify_alignment(meta)
    C.verify_coverage(meta, sd)
    C.verify_roundtrip(chunks, meta, sd, verbose=False)

    C.save(f'./zoo_chunks/{ck["model_id"]}', chunks, mask, seq_index, meta)
    rows.append(ck['model_id'])

assert len(rows) == len(set(rows)), 'duplicate model_id, checkpoints would overwrite each other'
print(f'saved {len(rows)} expert models to ./zoo_chunks')

extracting: 100%|██████████| 50/50 [01:48<00:00,  2.17s/it]

saved 50 expert models to ./zoo_chunks


### Backbone extracted separately, it is a reference model not zoo training data

In [ ]:
ck = torch.load('res_models/backbone_a.pt', map_location='cpu')
model = resnet20(num_classes=100)
model.load_state_dict(ck['state_dict'], strict=True)
sd = model.state_dict()

chunks, mask, seq_index, meta = C.extract_model(
    sd, 'backbone_a', checkpoint_path='res_models/backbone_a.pt',
    split_id='full', seed=ck['seed'], epoch=ck['epoch'], init_group='base_a'
)
C.verify_alignment(meta); C.verify_coverage(meta, sd)
C.verify_roundtrip(chunks, meta, sd, verbose=False)
C.save('./zoo_reference/backbone_a', chunks, mask, seq_index, meta)
print('backbone saved to ./zoo_reference/backbone_a')

backbone saved to ./zoo_reference/backbone_a
